## 10\_modele\_variables

# 10 · Variables explicatives du modèle

Construction du jeu d'apprentissage du modèle d'implantation Citiz.

1. Rattachement de chaque station Citiz à un carreau INSEE habité de 200 m (le sien, sinon le plus proche à moins de 300 m).
2. Périmètre : carreaux des communes comptant au moins une station Citiz (apprentissage), de l'Eurométropole de Strasbourg (test) et de la Métropole de Rouen (application).
3. Calcul de 12 variables dans un **rayon de 400 m** autour de chaque carreau : population et ménages, structure par âge, niveau de vie, ménages d'une personne, habitat collectif, propriétaires, ménages pauvres, logements sociaux, arrêts de transport (doublons à moins de 50 m fusionnés).

**Cible** : 1 si le carreau contient une station Citiz, 0 sinon.

**Sorties :** tables `carreaux_apprentissage` et `variables_rouen`.

In [ ]:
# 1. Dépendances
%pip install -q geopandas pyogrio py7zr

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# 2. Outils communs
import os, re, glob, shutil, tempfile, zipfile, warnings
import numpy as np
import pandas as pd
from pyproj import Transformer
warnings.filterwarnings("ignore")

BASE = "/lakehouse/default/Files"
if not os.path.isdir(BASE):
    raise RuntimeError("Lakehouse par défaut non attaché : attacher le Lakehouse mobilite au notebook.")
BRUT = os.path.join(BASE, "brut")
TRAITE = os.path.join(BASE, "traite")
CARTES = os.path.join(TRAITE, "cartes")
os.makedirs(CARTES, exist_ok=True)

# Conversions entre systèmes de coordonnées : 3035 = grille INSEE, 2154 = Lambert-93 (mètres), 4326 = GPS
T_3035_2154 = Transformer.from_crs(3035, 2154, always_xy=True)
T_3035_4326 = Transformer.from_crs(3035, 4326, always_xy=True)
T_4326_3035 = Transformer.from_crs(4326, 3035, always_xy=True)
T_4326_2154 = Transformer.from_crs(4326, 2154, always_xy=True)

def _copier_csv(df, chemin):
    """Écrit le CSV en local, puis le copie dans le Lakehouse. Écrire directement dans OneLake échoue parfois
    quand le fichier existe déjà (« Unable to rename file »)."""
    fd, tmp = tempfile.mkstemp(suffix=".csv")
    os.close(fd)
    try:
        df.to_csv(tmp, index=False)
        if os.path.exists(chemin):
            os.remove(chemin)
        shutil.copyfile(tmp, chemin)
    finally:
        os.remove(tmp)

def _adresse_table(nom):
    """Adresse OneLake de la table dans le Lakehouse attaché, dans le schéma dbo s'il existe."""
    import notebookutils
    ctx = notebookutils.runtime.context
    espace = ctx.get("defaultLakehouseWorkspaceId") or ctx.get("currentWorkspaceId")
    lakehouse = ctx.get("defaultLakehouseId")
    if not (espace and lakehouse):
        raise RuntimeError("Lakehouse par défaut introuvable : attacher le Lakehouse mobilite au notebook")
    schema = "/dbo" if os.path.isdir("/lakehouse/default/Tables/dbo") else ""
    return f"abfss://{espace}@onelake.dfs.fabric.microsoft.com/{lakehouse}/Tables{schema}/{nom}"

def _ecrire_table(df, nom):
    """Table Delta lue par Power BI. Noyau Spark : saveAsTable. Noyau Python : bibliothèque deltalake."""
    try:
        spark  # n'existe que dans le noyau Spark
    except NameError:
        import pyarrow as pa
        import notebookutils
        from deltalake import write_deltalake
        donnees = pa.Table.from_pandas(df, preserve_index=False)
        options = {"bearer_token": notebookutils.credentials.getToken("storage"), "use_fabric_endpoint": "true"}
        try:
            write_deltalake(_adresse_table(nom), donnees, mode="overwrite", schema_mode="overwrite",
                            storage_options=options)
        except TypeError:   # anciennes versions de deltalake
            write_deltalake(_adresse_table(nom), donnees, mode="overwrite", overwrite_schema=True,
                            storage_options=options)
        return
    spark.createDataFrame(df).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(nom)

def enregistrer(df, nom):
    """Enregistre un tableau en CSV dans Files/traite (relu par les autres notebooks)
    et en table du Lakehouse (lue par Power BI), avec le noyau Python comme avec le noyau Spark."""
    df = df.reset_index(drop=True).copy()
    _copier_csv(df, os.path.join(TRAITE, nom + ".csv"))
    for c in df.columns:
        if not (pd.api.types.is_numeric_dtype(df[c]) or pd.api.types.is_bool_dtype(df[c])):
            df[c] = df[c].map(lambda v: None if v is None or (isinstance(v, float) and np.isnan(v)) else str(v))
    try:
        _ecrire_table(df, nom)
        print(f"   table {nom} : {len(df)} lignes (copie dans Files/traite/{nom}.csv)")
    except Exception as e:
        print(f"   {nom}.csv : {len(df)} lignes. ATTENTION, table non créée pour Power BI : {e}")

def nb(n):
    """Écrit un nombre avec des espaces entre les milliers : 12 345."""
    return f"{n:,.0f}".replace(",", " ")

def lire(nom):
    """Relit un tableau enregistré par un autre notebook."""
    chemin = os.path.join(TRAITE, nom + ".csv")
    if not os.path.exists(chemin):
        raise FileNotFoundError(f"{nom}.csv introuvable dans Files/traite : exécuter d'abord le notebook qui le produit.")
    return pd.read_csv(chemin, dtype={"idcar": str, "commune": str, "idcar_1km": str, "candidat": str})

COLONNES_FILOSOFI = ["idcar_200m", "idcar_1km", "i_est_200", "lcog_geo", "ind", "men", "men_pauv", "men_1ind",
                     "men_5ind", "men_prop", "men_fmp", "ind_snv", "men_surf", "men_coll", "men_mais", "log_av45",
                     "log_45_70", "log_70_90", "log_ap90", "log_inc", "log_soc", "ind_0_3", "ind_4_5", "ind_6_10",
                     "ind_11_17", "ind_18_24", "ind_25_39", "ind_40_54", "ind_55_64", "ind_65_79", "ind_80p", "ind_inc"]
TEXTE_FILOSOFI = ["idcar_200m", "idcar_1km", "lcog_geo"]

def charger_filosofi():
    """Carreaux de 200 m de métropole (CSV ou GeoPackage), avec leur centre en coordonnées 3035."""
    csvs = sorted(glob.glob(os.path.join(BRUT, "filosofi", "**", "*.csv"), recursive=True),
                  key=os.path.getsize, reverse=True)
    csvs = [c for c in csvs if "met" in os.path.basename(c).lower()] or csvs
    if csvs:
        source = csvs[0]
        with open(source, encoding="utf-8", errors="replace") as f:
            entete = f.readline()
        sep = ";" if entete.count(";") > entete.count(",") else ","
        noms = [c.strip().strip('"') for c in entete.strip().split(sep)]
        utiles = [c for c in noms if c.lower() in COLONNES_FILOSOFI]
        df = pd.read_csv(source, sep=sep, usecols=utiles,
                         dtype={c: str for c in utiles if c.lower() in TEXTE_FILOSOFI})
    else:
        gpkgs = [g for g in glob.glob(os.path.join(BRUT, "**", "*.gpkg"), recursive=True)
                 if "carreaux" in os.path.basename(g).lower() and "met" in os.path.basename(g).lower()]
        if not gpkgs:
            raise FileNotFoundError("Carreaux Filosofi introuvables dans Files/brut/filosofi (notebook 01).")
        import pyogrio
        source = gpkgs[0]
        df = pyogrio.read_dataframe(source, read_geometry=False)
    df.columns = [c.lower() for c in df.columns]
    df = df[[c for c in COLONNES_FILOSOFI if c in df.columns]].copy()
    for c in df.columns:
        if c not in TEXTE_FILOSOFI and not pd.api.types.is_numeric_dtype(df[c]):   # décimales avec une virgule
            df[c] = pd.to_numeric(df[c].astype(str).str.replace(",", ".", regex=False), errors="coerce")
    nombres = [c for c in df.columns if c not in TEXTE_FILOSOFI]
    df[nombres] = df[nombres].fillna(0)
    # L'identifiant INSPIRE donne le coin sud-ouest du carreau en mètres : CRS3035RES200mN<nord>E<est>
    ne = df["idcar_200m"].astype(str).str.extract(r"CRS3035RES200mN(\d+)E(\d+)").astype(float)
    df["x3035"], df["y3035"] = ne[1] + 100, ne[0] + 100
    df = df.dropna(subset=["x3035", "y3035"])
    df["ix"] = ((df["x3035"] - 100) // 200).astype("int64")
    df["iy"] = ((df["y3035"] - 100) // 200).astype("int64")
    df["commune"] = df["lcog_geo"].astype(str).str[:5]
    df = df.rename(columns={"idcar_200m": "idcar"}).reset_index(drop=True)
    print(f"Filosofi : {nb(len(df))} carreaux de métropole lus ({os.path.basename(source)})")
    return df

def trouver_admin_express():
    """GeoPackage ADMIN EXPRESS : déjà extrait, ou extrait de l'archive .7z déposée dans Files/brut."""
    gpkgs = [g for g in glob.glob(os.path.join(BRUT, "**", "*.gpkg"), recursive=True)
             if "ADE" in os.path.basename(g).upper() or "ADMIN" in os.path.basename(g).upper()]
    if gpkgs:
        return gpkgs[0]
    archives = [a for a in glob.glob(os.path.join(BRUT, "**", "*.7z"), recursive=True) if "ADMIN" in a.upper()]
    if not archives:
        raise FileNotFoundError("Contours introuvables : fichiers IGN du notebook 01 ou archive ADMIN-EXPRESS (.7z) attendus dans Files/brut.")
    import py7zr
    with py7zr.SevenZipFile(sorted(archives)[-1], "r") as z:
        z.extractall(os.path.join(BRUT, "ign"))
    return trouver_admin_express()

def charger_territoire(court, mot):
    """Contour de l'intercommunalité (Lambert-93) et ses communes."""
    import geopandas as gpd
    f_epci = os.path.join(BRUT, f"epci_{court}.geojson")
    f_communes = os.path.join(BRUT, f"communes_{court}.geojson")
    if os.path.exists(f_epci) and os.path.exists(f_communes):
        epci = gpd.read_file(f_epci).to_crs(2154)
        communes = gpd.read_file(f_communes).to_crs(2154)
    else:
        gpkg = trouver_admin_express()
        epci = gpd.read_file(gpkg, layer="epci")
        epci = epci[epci["nom_officiel"].str.contains(mot, case=False, na=False)].to_crs(2154)
        siren = str(epci.iloc[0]["code_siren"])
        communes = gpd.read_file(gpkg, layer="commune")
        communes = communes[communes["codes_siren_des_epci"].astype(str).str.contains(siren, na=False)].to_crs(2154)
    contour = epci.geometry.iloc[0]
    communes = communes[["code_insee", "nom_officiel", "geometry"]].rename(
        columns={"code_insee": "commune", "nom_officiel": "commune_nom"})
    print(f"{epci.iloc[0]['nom_officiel']} : {len(communes)} communes, {contour.area / 1e6:.0f} km²")
    return contour, communes

def dans_contour(contour, x, y):
    """Vrai pour les points (Lambert-93) situés dans le contour."""
    import shapely
    xmin, ymin, xmax, ymax = contour.bounds
    resultat = np.zeros(len(x), dtype=bool)
    cadre = (x >= xmin) & (x <= xmax) & (y >= ymin) & (y <= ymax)
    shapely.prepare(contour)
    resultat[cadre] = shapely.contains_xy(contour, x[cadre], y[cadre])
    return resultat

In [ ]:
# 3. Carreaux INSEE et stations Citiz
fil = charger_filosofi()
stations = pd.read_csv(os.path.join(BRUT, "gbfs", "stations_citiz.csv"))
stations = stations.dropna(subset=["lat", "lon"])
sx, sy = T_4326_3035.transform(stations["lon"].values, stations["lat"].values)
stations["ix"], stations["iy"] = (sx // 200).astype("int64"), (sy // 200).astype("int64")
print(f"Stations Citiz : {len(stations)} sur {stations['reseau'].nunique()} réseaux")

Filosofi : 2 298 582 carreaux de métropole lus (carreaux_200m_met.csv)
Stations Citiz : 1615 sur 15 réseaux


In [ ]:
# 4. Rattacher chaque station à un carreau habité (le sien, sinon le plus proche à moins de 300 m)
from scipy.spatial import cKDTree
CLE = 1_000_000
index_grille = pd.Index(fil["ix"].values * CLE + fil["iy"].values)
pos = index_grille.get_indexer(stations["ix"].values * CLE + stations["iy"].values)
manquants = pos < 0
if manquants.any():
    arbre = cKDTree(np.column_stack([fil["x3035"].values, fil["y3035"].values]))
    d, proche = arbre.query(np.column_stack([sx[manquants], sy[manquants]]), distance_upper_bound=300)
    pos[manquants] = np.where(np.isfinite(d), proche, -1)
stations["pos"] = pos
retrouvees = stations[stations["pos"] >= 0]
print(f"Stations rattachées à un carreau habité : {len(retrouvees)} sur {len(stations)} "
      f"({100 * len(retrouvees) / len(stations):.0f} %)")

par_carreau = retrouvees.groupby("pos").agg(nb_stations=("station_id", "size"),
                                             capacite=("capacite", "sum"), reseau=("reseau", "first"))
fil["station"] = 0
fil.loc[par_carreau.index, "station"] = 1
fil["nb_stations"] = 0
fil.loc[par_carreau.index, "nb_stations"] = par_carreau["nb_stations"].values
fil["capacite_citiz"] = 0.0
fil.loc[par_carreau.index, "capacite_citiz"] = par_carreau["capacite"].fillna(0).values

# Communes « Citiz » : celles qui ont au moins une station, avec le ou les réseaux présents
fil_st = fil.loc[retrouvees["pos"].values, ["commune"]].assign(reseau=retrouvees["reseau"].values)
reseaux_commune = fil_st.groupby("commune")["reseau"].agg(lambda s: "|".join(sorted(set(s))))
print(f"Communes avec au moins une station Citiz : {len(reseaux_commune)}")

Stations rattachées à un carreau habité : 1604 sur 1615 (99 %)
Communes avec au moins une station Citiz : 310


In [ ]:
# 5. Strasbourg (test) et Rouen (application)
fil["x"], fil["y"] = T_3035_2154.transform(fil["x3035"].values, fil["y3035"].values)
contour_sxb, _ = charger_territoire("strasbourg", "Strasbourg")
contour_rouen, _ = charger_territoire("rouen", "Rouen")
fil["strasbourg"] = dans_contour(contour_sxb, fil["x"].values, fil["y"].values)
fil["rouen"] = dans_contour(contour_rouen, fil["x"].values, fil["y"].values)
fil["reseaux_commune"] = fil["commune"].map(reseaux_commune).fillna("")
fil["commune_citiz"] = fil["reseaux_commune"] != ""
print(f"Carreaux de l'Eurométropole : {int(fil['strasbourg'].sum())}, dont {int(fil.loc[fil['strasbourg'], 'station'].sum())} avec une station")
print(f"Carreaux de la Métropole de Rouen : {int(fil['rouen'].sum())}, dont {int(fil.loc[fil['rouen'], 'station'].sum())} avec une station")

Eurométropole de Strasbourg : 33 communes, 340 km²
Métropole Rouen Normandie : 71 communes, 664 km²
Carreaux de l'Eurométropole : 2776, dont 179 avec une station
Carreaux de la Métropole de Rouen : 5727, dont 0 avec une station


In [ ]:
# 6. Arrêts de transport : fichier national, doublons retirés (un arrêt par case de 50 m)
entete = pd.read_csv(os.path.join(BRUT, "arrets_france.csv"), nrows=0)
nat = pd.read_csv(os.path.join(BRUT, "arrets_france.csv"), usecols=["stop_lat", "stop_lon"], low_memory=False)
nat = nat.apply(pd.to_numeric, errors="coerce").dropna()
ax_, ay_ = T_4326_3035.transform(nat["stop_lon"].values, nat["stop_lat"].values)
cases = pd.DataFrame({"cx": (ax_ // 50).astype("int64"), "cy": (ay_ // 50).astype("int64")}).drop_duplicates()
print(f"Arrêts dans le fichier national : {nb(len(nat))}, après retrait des doublons : {nb(len(cases))}")
arrets_carreau = (cases.assign(ix=cases["cx"] // 4, iy=cases["cy"] // 4)
                       .groupby(["ix", "iy"]).size().rename("arrets"))
fil = fil.merge(arrets_carreau, left_on=["ix", "iy"], right_index=True, how="left")
fil["arrets"] = fil["arrets"].fillna(0)
del nat, cases

Arrêts dans le fichier national : 637 981, après retrait des doublons : 246 139


In [ ]:
# 7. Variables dans un rayon de 400 m (le carreau et ses voisins dont le centre est à 400 m ou moins)
VOISINS = [(dx, dy) for dx in range(-2, 3) for dy in range(-2, 3) if (dx * dx + dy * dy) * 200 ** 2 <= 400 ** 2]
gardes = fil[fil["commune_citiz"] | fil["strasbourg"] | fil["rouen"]].copy()
cles = gardes["ix"].values * CLE + gardes["iy"].values
SOMMES = ["ind", "men", "ind_25_39", "ind_65_79", "ind_80p", "ind_0_3", "ind_4_5", "ind_6_10", "ind_11_17",
          "ind_snv", "men_1ind", "men_coll", "men_prop", "men_pauv", "log_soc",
          "log_av45", "log_45_70", "log_70_90", "log_ap90", "log_inc"]
grille = {c: fil[c].values for c in SOMMES}
grille_arrets = fil["arrets"].values
total = {c: np.zeros(len(gardes)) for c in SOMMES}
total_arrets = np.zeros(len(gardes))
for dx, dy in VOISINS:
    p = index_grille.get_indexer(cles + dx * CLE + dy)
    ok = p >= 0
    for c in SOMMES:
        total[c][ok] += grille[c][p[ok]]
    total_arrets[ok] += grille_arrets[p[ok]]

def part(a, b):
    return np.where(b > 0, a / np.maximum(b, 1e-9), 0.0)
logements = sum(total[c] for c in ["log_av45", "log_45_70", "log_70_90", "log_ap90", "log_inc"])
gardes["pop_400"] = total["ind"]
gardes["menages_400"] = total["men"]
gardes["part_25_39"] = part(total["ind_25_39"], total["ind"])
gardes["part_65_plus"] = part(total["ind_65_79"] + total["ind_80p"], total["ind"])
gardes["part_moins_18"] = part(total["ind_0_3"] + total["ind_4_5"] + total["ind_6_10"] + total["ind_11_17"], total["ind"])
gardes["niveau_vie_400"] = part(total["ind_snv"], total["ind"])
gardes["part_1_personne"] = part(total["men_1ind"], total["men"])
gardes["part_collectif"] = part(total["men_coll"], total["men"])
gardes["part_proprietaires"] = part(total["men_prop"], total["men"])
gardes["part_pauvres"] = part(total["men_pauv"], total["men"])
gardes["part_logement_social"] = part(total["log_soc"], logements)
gardes["arrets_400"] = total_arrets
VARIABLES = ["pop_400", "menages_400", "part_25_39", "part_65_plus", "part_moins_18", "niveau_vie_400",
             "part_1_personne", "part_collectif", "part_proprietaires", "part_pauvres", "part_logement_social", "arrets_400"]
print(f"{len(VARIABLES)} variables calculées pour {len(gardes)} carreaux")
print(gardes[VARIABLES].describe().T[["mean", "min", "max"]].round(2).to_string())

12 variables calculées pour 72070 carreaux
                          mean       min       max
pop_400                1512.71      1.00  14456.50
menages_400             739.91      0.30   7970.00
part_25_39                0.18      0.00      0.67
part_65_plus              0.21      0.00      0.72
part_moins_18             0.20      0.00      0.52
niveau_vie_400        26420.41  11135.93  58035.01
part_1_personne           0.36      0.00      0.97
part_collectif            0.48      0.00      1.00
part_proprietaires        0.59      0.00      1.00
part_pauvres              0.14      0.00      0.80
part_logement_social      0.16      0.00      0.99
arrets_400                4.36      0.00     47.00


In [ ]:
# 8. Enregistrer : exemples (villes Citiz et Strasbourg) et carreaux de Rouen
gardes["lon"], gardes["lat"] = T_3035_4326.transform(gardes["x3035"].values, gardes["y3035"].values)
gardes["grand_est"] = gardes["reseaux_commune"].str.contains("citiz_grand_est")
colonnes = ["idcar", "commune", "lat", "lon", "ix", "iy", "ind", "i_est_200", "station", "nb_stations", "capacite_citiz",
            "reseaux_commune", "grand_est", "strasbourg"] + VARIABLES
apprentissage = gardes[gardes["commune_citiz"] | gardes["strasbourg"]][colonnes]
print("Enregistrement :")
enregistrer(apprentissage, "carreaux_apprentissage")
enregistrer(gardes[gardes["rouen"]][colonnes], "variables_rouen")
print(f"\nExemples : {len(apprentissage)} carreaux, dont {int(apprentissage['station'].sum())} avec une station Citiz")
print("Carreaux avec station, par réseau :")
print(apprentissage[apprentissage["station"] == 1]["reseaux_commune"].str.split("|").explode().value_counts().to_string())

Enregistrement :
   carreaux_apprentissage.csv : 66343 lignes. ATTENTION, table non créée pour Power BI : Unsupported reader features required: [DeletionVectors]


[2026-10-21T08:24:37Z WARN  delta_kernel::listed_log_files] Found file 00000000000000000000.parquet with unknown file type Unknown at version 0
